In [ ]:
import gc
from pathlib import Path

import polars as pl
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder

# Configure sklearn transform
from sklearn import set_config
set_config(transform_output="polars")

# Configure Polars print outs
pl.Config.set_tbl_rows(30)  # -1 means display all rows
pl.Config.set_tbl_cols(-1)  # -1 means display all columns

In [ ]:
INPUT_DATA_DIR = Path("./data")
RAW_INPUT_DATA_DIR = INPUT_DATA_DIR / "raw"
FEATURES_DATA_PATH = INPUT_DATA_DIR / "m5-acc-features-df.pq"

# Feature selection
SALES_LAGS = [1, 2, 3, 4, 7, 14]
ROLLING_SALES_WINDOWS = [2, 3, 4, 7, 14]
CYCLICAL_CALENDAR_FEATURE_SUFFIXES = ["wday", "month"]
N_EVENTS = 2

CATEGORICAL_FEATURE_COLS = [
    "item_id", "dept_id", "cat_id", "store_id", "state_id",
    *[f"event_name_{i}" for i in range(1, N_EVENTS + 1)],
    *[f"event_type_{i}" for i in range(1, N_EVENTS + 1)],
    *[f"last_event_type_{i}" for i in range(1, N_EVENTS + 1)],
    *[f"next_event_type_{i}" for i in range(1, N_EVENTS + 1)],
]
CONTINUOUS_FEATURE_COLS = [
    "sell_price",
    *[f"sales_lagged_{lag}" for lag in SALES_LAGS],
    *[f"sales_rolling_mean_{w}" for w in ROLLING_SALES_WINDOWS],
    *[f"{trig}_{suffix}" for suffix in CYCLICAL_CALENDAR_FEATURE_SUFFIXES for trig in ("sin", "cos")],
    "is_weekend",
    *[f"days_since_event_{i}" for i in range(1, N_EVENTS + 1)],
    *[f"days_until_event_{i}" for i in range(1, N_EVENTS + 1)],
]
ALL_FEATURE_COLS = CATEGORICAL_FEATURE_COLS + CONTINUOUS_FEATURE_COLS
LABEL_COL_NAME = "sales"

# Validation config
VALIDATION_DAYS = 28

### Load Data

In [ ]:
feature_df = pl.read_parquet(FEATURES_DATA_PATH)

In [ ]:
# For submission will need to construct features iteratively.
# Keep the final slice of the train data needed to construct features for the
# first submission timestamp, and store as a separate df so the full feature df
# isn't needed at inference.

max_lookback = max(SALES_LAGS + ROLLING_SALES_WINDOWS)
final_feature_df_slice = (
    feature_df.select(["id", "d_index", "sales"])
    .sort(by=["id", "d_index"])
    .group_by("id")
    .tail(max_lookback)
    .sort(by=["id", "d_index"])
)

final_feature_df_slice.tail(max_lookback * 2 + 2)

## Prepare data

Numeric features are cast to float32 (nulls become NaN, which HistGradientBoosting handles natively). Categoricals stay as pandas `category` dtype.

In [ ]:
feature_df_slice = (
    feature_df
    .sort(by=["id", "d_index"])
    .group_by("id", maintain_order=True)
    .tail(1000)
)

In [ ]:
# Select features and cast to correct types

X, y = feature_df_slice.select(ALL_FEATURE_COLS), feature_df_slice.select(LABEL_COL_NAME)

X = (
    X.cast({
        col_name: pl.Categorical
        for col_name in CATEGORICAL_FEATURE_COLS
    })
    .cast({
        col_name: pl.Float32
        for col_name in CONTINUOUS_FEATURE_COLS
    })
)
y = y.cast(pl.Float32)

X.shape, y.shape

## Model

In [ ]:
target_encoding_cols = ["item_id"]
target_encoder = TargetEncoder(target_type="continuous")

model = Pipeline(
    steps=[
        (
            "preprocessor",
            ColumnTransformer(
                [("item_encoder", target_encoder, target_encoding_cols)],
                remainder="passthrough",
                verbose_feature_names_out=False,
            )
        ),
        (
            "regressor",
            HistGradientBoostingRegressor(
                max_iter=10,
                categorical_features="from_dtype",
                random_state=0,
        )),
    ]
)
model

In [ ]:
model.fit(X, y.to_series())

## Submission

In [ ]:
# Load data needed for inference

# Calendar data
calendar_schema_overrides = {
    "wm_yr_wk": pl.Int16,
    **{date_type: pl.Int16 for date_type in ["wday", "month", "year"]},
    **{f"snap_{state}": pl.Int16 for state in ["CA", "TX", "WI"]}
}
raw_calendar_df = pl.read_csv(
    RAW_INPUT_DATA_DIR / "calendar.csv",
    try_parse_dates=True,
    schema_overrides=calendar_schema_overrides,
)

# Sell prices
sell_price_schema_overrides = {
    "wm_yr_wk": pl.Int16,
    "sell_price": pl.Float32,
}
raw_sell_price_df = pl.read_csv(
    RAW_INPUT_DATA_DIR / "sell_prices.csv",
    schema_overrides=sell_price_schema_overrides
)
sample_submission_df = pl.read_csv(
    RAW_INPUT_DATA_DIR / "sample_submission.csv"
)

In [ ]:
# Need to recompute some calendar and event features for inference

event_features_df = (
    raw_calendar_df
    .select(["wm_yr_wk", "d", "date", "event_name_1", "event_type_1", "event_name_2", "event_type_2"])
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        pl.when(pl.col(f"event_name_{i}").is_not_null())
        .then(pl.col("d_index"))
        .otherwise(None)
        .alias(f"event_index_{i}")
        for i in (1, 2)
    )
    # Days until next / since last event
    .with_columns(
        # Days since last event
        *[
            (pl.col("d_index") - pl.col(f"event_index_{i}").forward_fill())
            .alias(f"days_since_event_{i}")
            for i in (1, 2)
        ],
        # Last event type
        *[
            (pl.col(f"event_type_{i}").forward_fill()
            .alias(f"last_event_type_{i}"))
            for i in range(1, N_EVENTS + 1)
        ],
        # Days until next event
        *[
            (pl.col(f"event_index_{i}").backward_fill() - pl.col("d_index"))
            .alias(f"days_until_event_{i}")
            for i in (1, 2)
        ],
        # Next event type
        *[
            (pl.col(f"event_type_{i}").backward_fill()
            .alias(f"next_event_type_{i}"))
            for i in range(1, N_EVENTS + 1)
        ],
    )
    .with_columns(
        # Clip days since last event and event type to 7 days.
        *[
            pl.when(pl.col(f"days_since_event_{i}") <= 7)
            .then(pl.col(f"days_since_event_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.when(pl.col(f"days_since_event_{i}") <= 7).
            then(pl.col(f"last_event_type_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        # Clip days until next event and event type to 7 days.
        *[              
            pl.when(pl.col(f"days_until_event_{i}") <= 7)
            .then(pl.col(f"days_until_event_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.when(pl.col(f"days_until_event_{i}") <= 7)
            .then(pl.col(f"next_event_type_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ]
    )
)

calendar_features_df = (
    raw_calendar_df
    .select(["wm_yr_wk", "d", "date", "wday", "month", "year"])
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        sin_wday=(2 * np.pi * pl.col("wday") / 7).sin().cast(pl.Float32),
        cos_wday=(2 * np.pi * pl.col("wday") / 7).cos().cast(pl.Float32),
        sin_month=(2 * np.pi * pl.col("month") / 12).sin().cast(pl.Float32),
        cos_month=(2 * np.pi * pl.col("month") / 12).cos().cast(pl.Float32),
        is_weekend=(pl.col("wday") <= 2).cast(pl.Float32),
    )
)

In [ ]:
FORECAST_HORIZON = 28
MAX_TRAIN_INDEX = 1941
SUBMISSION_F_COLS = [f"F{i}" for i in range(1, FORECAST_HORIZON + 1)]

_SERIES_ID_RE = (
    r"^(?P<cat>[A-Z]+)_(?P<dept>\d+)_(?P<item>\d+)_(?P<state>[A-Z]+)_(?P<store>\d+)"
    r"_(?:validation|evaluation)$"
)

FORECAST_IDS = (
    sample_submission_df
    .filter(sample_submission_df["id"].str.ends_with("_evaluation"))
    ["id"]
    .unique()
    .to_list()
)

In [ ]:
# Iterate over forecast timestamps, save forecasts, and
# construct features for next timestamp.

all_forecast_dfs: list[pl.DataFrame] = []

for f_index in range(1, FORECAST_HORIZON + 1):
    d_index = MAX_TRAIN_INDEX + f_index
    
    # Construct test_df with identifier columns
    test_df = (
        pl.DataFrame({"id": FORECAST_IDS})
        .with_columns(parsed_id=pl.col("id").str.extract_groups(_SERIES_ID_RE))
        .with_columns(
            item_id=pl.format(
                "{}_{}_{}",
                pl.col("parsed_id").struct.field("cat"),
                pl.col("parsed_id").struct.field("dept"),
                pl.col("parsed_id").struct.field("item"),
            ),
            dept_id=pl.format(
                "{}_{}",
                pl.col("parsed_id").struct.field("cat"),
                pl.col("parsed_id").struct.field("dept"),
            ),
            cat_id=pl.format(
                "{}",
                pl.col("parsed_id").struct.field("cat"),
            ),
            store_id=pl.format(
                "{}_{}",
                pl.col("parsed_id").struct.field("state"),
                pl.col("parsed_id").struct.field("store"),
            ),
            state_id=pl.format(
                "{}",
                pl.col("parsed_id").struct.field("state"),
            )
        )
        .drop("parsed_id")
        .with_columns(
            id=pl.col("id").cast(pl.Categorical),
            d_index=pl.lit(d_index).cast(pl.Int16),
            sales=pl.lit(None).cast(pl.Int16)
        )
    )

    # Construct lagged sales features
    previous_sales_df = pl.concat(
        [
            final_feature_df_slice,
            *[
                df
                .select(["id", "d_index", "sales"])
                .with_columns(id=pl.col("id").cast(pl.Categorical))
                for df in all_forecast_dfs
            ],
            test_df.select(["id", "d_index", "sales"])
        ]
    )
    
    test_sales_lag_df = (
        previous_sales_df
        .sort(by=["id", "d_index"])
        .with_columns(
            pl.col("sales").shift(lag).over("id", order_by="d_index").alias(f"sales_lagged_{lag}")
            for lag in SALES_LAGS
        )
    )

    # Construct rolling window features
    test_rolling_window_df = (
        previous_sales_df
        .sort(by=["id", "d_index"])
        .with_columns(
            pl.col("sales")
            .shift(1)
            .rolling_mean(window_size=window, min_samples=window)
            .over("id", order_by="d_index")
            .alias(f"sales_rolling_mean_{window}")
            .cast(pl.Float32)
            for window in ROLLING_SALES_WINDOWS
        )
    )

    # Join id, lagged, rolling, price, and calendar dfs
    test_df = (
        test_df.drop("sales")
        .join(
            other=test_sales_lag_df.drop("sales"),
            on=["id", "d_index"],
            how="inner",
            validate="1:1",
        )
        .join(
            other=test_rolling_window_df.drop("sales"),
            on=["id", "d_index"],
            how="inner",
            validate="1:1",
        )
        .join(
            other=(
                calendar_features_df.drop("d", "date")
                .join(
                    other=event_features_df.drop("d", "date"),
                    on=["wm_yr_wk", "d_index"],
                    how="inner",
                    validate="1:1",
                )
            ),
            on="d_index",
            how="inner",
            validate="m:1"
        )
        .join(
            other=raw_sell_price_df,
            on=["item_id", "store_id", "wm_yr_wk"],
            how="inner",
            validate="1:1"
        )
    )

    # Make predictions
    X_test = (
        test_df
        .select(ALL_FEATURE_COLS)
        .cast({
            col_name: pl.Categorical
            for col_name in CATEGORICAL_FEATURE_COLS
        })
        .cast({
            col_name: pl.Float32
            for col_name in CONTINUOUS_FEATURE_COLS
        })
    )
    y_hat = model.predict(X_test)

    # Predictions are aligned with the rows of test_df (not FORECAST_IDS, as the
    # joins don't guarantee row order). Sales can't be negative, and are rounded
    # to integers so they can be fed back in as lagged sales features.
    y_hat_df = (
        pl.DataFrame({"id": test_df["id"].cast(pl.String), "sales": y_hat})
        .with_columns(
            sales=pl.col("sales").clip(lower_bound=0).round().cast(pl.Int16),
            d_index=pl.lit(d_index).cast(pl.Int16),
            F=pl.format("F{}", f_index),
        )
    )

    # Save predictions
    all_forecast_dfs.append(y_hat_df)


In [ ]:
final_forecast_df = (
    pl.concat(all_forecast_dfs)
    .with_columns(sales=pl.col("sales").cast(pl.Int64))
    .pivot(on="F", index="id", values="sales")
)

submission_df = (
    sample_submission_df
    .join(
        other=final_forecast_df,
        on="id",
        how="left",
        suffix="_model",
        validate="1:1"
    )
    .with_columns(
        pl.coalesce(pl.col(f"{c}_model"), pl.col(c)).cast(pl.Int64).alias(c)
        for c in SUBMISSION_F_COLS
    )
    .select(sample_submission_df.columns)
)

In [ ]:
submission_df.tail()